# RouteFinder — Training on Kaggle

**Before running:**
1. Enable **Internet** (Settings → Internet → On)
2. Enable **GPU** (Settings → Accelerator → GPU T4 x1)
3. Add two secrets (Add-ons → Secrets → Add new secret), then toggle both **on** for this notebook:
   - `HF_TOKEN` — your HuggingFace token
   - `GITHUB_TOKEN` — a GitHub fine-grained PAT with read-only **Contents** permission on this repo

Then click **Run All**. The best checkpoint is saved to `/kaggle/working/checkpoints/` and zipped for download at the end.

In [ ]:
# ── 1. Install dependencies not pre-installed on Kaggle ───────────────────────
!pip install -q lightly pytorch-metric-learning
# timm, pytorch-lightning, datasets, torch are already on Kaggle

In [ ]:
# ── 2. Pull the repo and add it to the Python path ───────────────────────────
import os, sys, importlib
from kaggle_secrets import UserSecretsClient

gh_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{gh_token}@github.com/Declan-Bracken/RouteFinder.git"
REPO_DIR = "/kaggle/working/RouteFinder"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

# Add the train/ subdirectory so train.py is importable directly as `train`
TRAIN_DIR = os.path.join(REPO_DIR, "train")
if TRAIN_DIR not in sys.path:
    sys.path.insert(0, TRAIN_DIR)

# Bust the module cache so re-runs after a git pull pick up the new code
for mod in [k for k in sys.modules if "train" in k]:
    del sys.modules[mod]

print("Repo ready.")

In [ ]:
# ── 3. Load HF token from Kaggle Secrets ──────────────────────────────────────
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token loaded:", hf_token[:8] + "...")

In [ ]:
# ── 3b. Download training images from B2 ─────────────────────────────────────
# The manifest CSV is already in the repo at data/manifest.csv.
# This cell just downloads the actual images from B2 to local disk.
# Add these four secrets in Kaggle (Add-ons → Secrets):
#   B2_KEY_ID, B2_APPLICATION_KEY, B2_BUCKET_NAME, B2_ENDPOINT_URL
import os, boto3, pandas as pd
from botocore.config import Config as BotoConfig
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm

secrets = UserSecretsClient()
b2 = boto3.client(
    "s3",
    endpoint_url          = secrets.get_secret("B2_ENDPOINT_URL"),
    aws_access_key_id     = secrets.get_secret("B2_KEY_ID"),
    aws_secret_access_key = secrets.get_secret("B2_APPLICATION_KEY"),
    config=BotoConfig(signature_version="s3v4"),
)
B2_BUCKET    = secrets.get_secret("B2_BUCKET_NAME")
MANIFEST_PATH = os.path.join(REPO_DIR, "data", "manifest.csv")
IMAGE_DIR     = "/kaggle/working/images"
os.makedirs(IMAGE_DIR, exist_ok=True)

manifest_df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest: {len(manifest_df)} images across {manifest_df['route_id'].nunique()} routes")

skipped = 0
for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Downloading"):
    dest = os.path.join(IMAGE_DIR, f"{row['image_id']}.jpg")
    if os.path.exists(dest):
        skipped += 1
        continue
    b2.download_file(Bucket=B2_BUCKET, Key=row["b2_key"], Filename=dest)

print(f"Done — {len(manifest_df) - skipped} downloaded, {skipped} already cached")

In [ ]:
# ── 4. Configure the run ──────────────────────────────────────────────────────
# Edit anything here. Everything else in train.py has sensible defaults.
from train import Config

cfg = Config(
    hf_token          = hf_token,
    hf_dataset        = "DeclanBracken/RouteFinderDatasetV2",

    # Model — start with frozen backbone (num_unfrozen_blocks=0).
    # Increase to 1 or 2 once you have more field data.
    num_unfrozen_blocks = 0,

    # Training
    batch_size        = 128,
    lr                = 1e-3,
    temperature       = 0.07,
    max_epochs        = 100,
    warmup_epochs     = 5,
    patience          = 15,

    # Checkpoints land in /kaggle/working/checkpoints/
    checkpoint_dir    = "/kaggle/working/checkpoints",
)

print(cfg)

In [ ]:
# ── 5. Train ──────────────────────────────────────────────────────────────────
from train import train

best_ckpt, metrics_csv = train(cfg)
print(f"\nBest checkpoint: {best_ckpt}")

In [ ]:
# ── 5b. Training curves ───────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(metrics_csv)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

train_loss = df.dropna(subset=["train_loss"])
val_loss   = df.dropna(subset=["val_loss"])
ax1.plot(train_loss["epoch"], train_loss["train_loss"], label="train")
ax1.plot(val_loss["epoch"],   val_loss["val_loss"],     label="val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss"); ax1.legend()

recall = df.dropna(subset=["val_recall@1"])
ax2.plot(recall["epoch"], recall["val_recall@1"], marker="o")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Recall@1"); ax2.set_title("Val Recall@1")

plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Quick eval on val split ────────────────────────────────────────────────
import torch, random
from torch.utils.data import DataLoader
from sklearn.neighbors import NearestNeighbors
from datasets import load_dataset
from train import RouteFinderModel, EvalDataset

device = "cuda" if torch.cuda.is_available() else "cpu"

ds = load_dataset(cfg.hf_dataset, token=cfg.hf_token)
all_routes = list(set(ds["train"]["label"]))
random.seed(42)
n_test = max(1, int(len(all_routes) * cfg.test_split))
test_routes = set(random.sample(all_routes, n_test))
test_split = ds["train"].filter(lambda x: x["label"] in test_routes)

model = RouteFinderModel.load_from_checkpoint(best_ckpt).to(device).eval()
loader = DataLoader(EvalDataset(test_split), batch_size=64, num_workers=2)

embeddings, labels = [], []
with torch.no_grad():
    for imgs, lbls in loader:
        embeddings.append(model(imgs.to(device)).cpu())
        labels.extend(lbls.tolist())

emb = torch.cat(embeddings).numpy()
knn = NearestNeighbors(n_neighbors=11, metric="cosine")
knn.fit(emb)
_, indices = knn.kneighbors(emb)

print("\n── Final Recall@K ──")
for k in [1, 3, 5, 10]:
    hits = sum(labels[i] in [labels[j] for j in indices[i][1:k+1]] for i in range(len(labels)))
    print(f"  Recall@{k}: {hits/len(labels):.4f}")

In [ ]:
# ── 6b. Retrieval visualization ───────────────────────────────────────────────
import random
import torch
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from train import RouteFinderModel, EVAL_TRANSFORM

N_QUERIES = 6   # rows
K         = 5   # neighbours shown per query

model.eval()
raw_images, viz_labels, viz_emb = [], [], []

with torch.no_grad():
    for i in range(len(test_split)):
        sample = test_split[i]
        img = sample["image"].convert("RGB")
        raw_images.append(img)
        viz_labels.append(sample["label"])
        viz_emb.append(model(EVAL_TRANSFORM(img).unsqueeze(0).to(device)).cpu())

viz_emb = torch.cat(viz_emb).numpy()
knn_v = NearestNeighbors(n_neighbors=K + 1, metric="cosine").fit(viz_emb)
_, nn_idx = knn_v.kneighbors(viz_emb)

# Only query images that have at least one other image of the same route
eligible = [i for i in range(len(viz_labels))
            if sum(l == viz_labels[i] for l in viz_labels) > 1]
query_idxs = random.sample(eligible, min(N_QUERIES, len(eligible)))

fig, axes = plt.subplots(len(query_idxs), K + 1, figsize=(3 * (K + 1), 3 * len(query_idxs)))
if len(query_idxs) == 1:
    axes = [axes]

for row, qi in enumerate(query_idxs):
    for col, idx in enumerate([qi] + list(nn_idx[qi][1:K + 1])):
        ax = axes[row][col]
        ax.imshow(raw_images[idx])
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            color, label = "royalblue", "Query"
        else:
            color = "green" if viz_labels[idx] == viz_labels[qi] else "red"
            label = f"#{col} ✓" if color == "green" else f"#{col} ✗"
        ax.set_title(label, fontsize=8, color=color, fontweight="bold")
        ax.add_patch(plt.Rectangle(
            [0, 0], 1, 1, fill=False, color=color, linewidth=4,
            transform=ax.transAxes, clip_on=False
        ))

plt.suptitle(f"Top-{K} Retrieval — green = correct route, red = wrong", y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. Zip the best checkpoint for download ───────────────────────────────────
import zipfile
from IPython.display import FileLink

zip_path = "/kaggle/working/routefinder_best.zip"
ckpt_name = best_ckpt.split("/")[-1]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(best_ckpt, arcname=ckpt_name)

print(f"Saved: {zip_path}")
FileLink("routefinder_best.zip")